# Chapter 16: Interactive Charts with plotly express

Companion notebook for *Master Data Visualization: A Hands-On Beginner's Guide with Python, Matplotlib, Seaborn & Plotly*.

Run the cells from top to bottom (**Kernel → Restart Kernel and Run All Cells**). The data files this chapter uses are in the same folder as this notebook.

## 16.1 What "Interactive" Means

### `px.scatter()`

In [1]:
import pandas as pd
import plotly.express as px

stores = pd.read_csv('stores.csv')
fig = px.scatter(stores, x='visitors', y='daily_revenue',
                 color='region')
fig.show()

In [2]:
type(fig)

plotly.graph_objs._figure.Figure

## 16.2 Scatter Plots: Looks, Labels and Hover

### Template and colors

In [3]:
colors = ['#0b3a47', '#b26a00', '#6fb3c2', '#f0c987']
fig = px.scatter(stores, x='visitors', y='daily_revenue',
                 color='region', template='simple_white',
                 color_discrete_sequence=colors)
fig.show()

### `labels` and `hover_data`

In [4]:
fig = px.scatter(
    stores, x='visitors', y='daily_revenue', color='region',
    symbol='region', hover_name='store',
    hover_data={'region': False, 'conversion': ':.1%'},
    labels={'visitors': 'Visitors per day',
            'daily_revenue': 'Revenue per day ($)',
            'conversion': 'Conversion', 'region': 'Region'},
    template='simple_white', color_discrete_sequence=colors)
fig.update_traces(marker_size=10)
fig.show()

## 16.3 Lines and Areas

In [5]:
daily = pd.read_csv('daily_sales.csv', parse_dates=['date'])
daily.head(3)

,date,revenue,orders,target
0,2025-01-06,1740,29,1900
1,2025-01-07,1560,24,1900
2,2025-01-08,1870,32,1900


### `px.line()`

In [6]:
fig = px.line(daily, x='date', y='revenue',
              labels={'date': '', 'revenue': 'Revenue per day ($)'},
              template='simple_white', color_discrete_sequence=colors)
fig.show()

### `fig.update_xaxes()` and `fig.add_hline()`

In [7]:
average = daily['revenue'].mean()
round(average)

2175

In [8]:
fig.update_xaxes(rangeslider_visible=True)
fig.add_hline(y=average, line_width=2, line_dash='dash',
              line_color='#b26a00', opacity=1,
              annotation_text='Average: $2,175',
              annotation_position='bottom right',
              annotation_bgcolor='white')
fig.show()

### `px.area()`

In [9]:
channels = pd.read_csv('channel_monthly.csv')
fig = px.area(channels, x='month', y=['Online', 'In store', 'Phone'],
              labels={'month': '', 'value': 'Revenue per month ($)',
                      'variable': 'Channel'},
              template='simple_white', color_discrete_sequence=colors)
fig.show()

## 16.4 Bar Charts and Category Order

In [10]:
orders = pd.read_csv('orders.csv')
product_totals = orders.groupby('product', as_index=False)[
    'order_value'].sum()
product_totals

,product,order_value
0,Headphones,1316
1,Keyboard,1248
2,Monitor,2536
3,Mouse,677
4,Webcam,1460


### `px.bar()`

In [11]:
fig = px.bar(product_totals, x='product', y='order_value',
             labels={'product': '', 'order_value': 'Revenue ($)'},
             template='simple_white', color_discrete_sequence=colors)
fig.show()

### `category_orders`

In [12]:
best_first = (product_totals
              .sort_values('order_value', ascending=False)['product']
              .tolist())
fig = px.bar(product_totals, x='product', y='order_value',
             category_orders={'product': best_first},
             labels={'product': '', 'order_value': 'Revenue ($)'},
             template='simple_white', color_discrete_sequence=colors)
fig.show()

## 16.5 Histograms, Box Plots and Panels

### `px.histogram()`

In [13]:
fig = px.histogram(orders, x='order_value', nbins=15,
                   labels={'order_value': 'Order value ($)'},
                   template='simple_white',
                   color_discrete_sequence=colors)
fig.show()

### `facet_col`

In [14]:
fig = px.histogram(
    orders, x='order_value', nbins=15, facet_col='channel',
    category_orders={'channel': ['Online', 'In store', 'Phone']},
    labels={'order_value': 'Order value ($)', 'channel': 'Channel'},
    template='simple_white', color_discrete_sequence=colors)
fig.show()

### `px.box()`

In [15]:
fig = px.box(orders, x='product', y='order_value',
             category_orders={'product': best_first},
             labels={'product': '', 'order_value': 'Order value ($)'},
             template='simple_white', color_discrete_sequence=colors)
fig.show()

## 16.6 Pie Charts

### `px.pie()`

In [16]:
channel_totals = orders.groupby('channel', as_index=False)[
    'order_value'].sum()
fig = px.pie(channel_totals, names='channel', values='order_value',
             template='simple_white', color_discrete_sequence=colors)
fig.update_traces(texttemplate='%{percent:.0%}')
fig.show()

## 16.7 Finishing Touches

### `fig.update_traces()`

In [17]:
fig = px.bar(product_totals, x='product', y='order_value',
             category_orders={'product': best_first},
             labels={'product': '', 'order_value': 'Revenue ($)'},
             template='simple_white')
fig.update_traces(marker_color=[  # one color per row of product_totals
    '#0b3a47' if p == 'Monitor' else '#c4c9ce'
    for p in product_totals['product']])
fig.show()

### `fig.update_layout()`

In [18]:
fig.update_layout(
    title='Monitors bring in the most revenue',
    yaxis_tickprefix='$', yaxis_tickformat=',',
    width=600, height=400)
fig.show()

### `fig.add_annotation()`

In [19]:
revenue = product_totals.set_index('product')['order_value']
share = revenue['Monitor'] / revenue.sum()
count = (orders['product'] == 'Monitor').sum()
fig.add_annotation(x='Monitor', y=revenue['Monitor'], ax=120, ay=-25,
                   text=f'{count} orders, {share:.0%} of revenue',
                   showarrow=True, arrowhead=2)
fig.show()

## 16.8 Saving and Sharing

### `fig.write_html()`

In [20]:
fig = px.line(daily, x='date', y='revenue',
              labels={'date': '', 'revenue': 'Revenue per day ($)'},
              template='simple_white', color_discrete_sequence=colors)
fig.update_xaxes(rangeslider_visible=True)
fig.write_html('daily_revenue.html')